In [0]:
%pip install databricks-langchain langchain langgraph mlflow
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Connect to Databricks Built-in LLM

from databricks_langchain import ChatDatabricks

# Connect to Databricks built-in LLM
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1
)

print(" Mosaic AI LLM connected!")
response = llm.invoke("Say hello in one sentence!")
print(f"LLM Response: {response.content}")

 Mosaic AI LLM connected!
LLM Response: Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [0]:

# Tools for Agent to read Bronze Delta tables

from langchain_core.tools import tool

@tool
def get_sales_data(question: str) -> str:
    """Use this tool to get sales data from bronze_sales_data table"""
    df = spark.sql("SELECT * FROM bronze_sales_data LIMIT 1000")
    data = df.toPandas().to_string()
    return f"Sales Data:\n{data}"

@tool
def get_api_users(question: str) -> str:
    """Use this tool to get user data from bronze_api_users table"""
    df = spark.sql("SELECT * FROM bronze_api_users LIMIT 1000")
    data = df.toPandas().to_string()
    return f"API Users Data:\n{data}"

@tool
def get_audit_log(question: str) -> str:
    """Use this tool to get ingestion audit log from bronze_audit_log table"""
    df = spark.sql("SELECT * FROM bronze_audit_log LIMIT 1000")
    data = df.toPandas().to_string()
    return f"Audit Log:\n{data}"

print("Tools created!")
print("get_sales_data")
print("get_api_users")
print("get_audit_log")

Tools created!
get_sales_data
get_api_users
get_audit_log


In [0]:

# Build Mosaic AI Agent with LangGraph

from langgraph.prebuilt import create_react_agent

# Create agent with tools
tools = [get_sales_data, get_api_users, get_audit_log]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful data analyst assistant. You have access to Bronze Delta tables containing sales, user and audit data. Always use the appropriate tool to fetch data before answering questions."
)

print("Mosaic AI Agent created!")
print("Agent has access to 3 Bronze Delta tables!")

Mosaic AI Agent created!
Agent has access to 3 Bronze Delta tables!


/home/spark-03a28ca8-1d88-4e55-bbb8-f0/.ipykernel/18293/command-8476672931950047-4218870950:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [0]:
# Test Mosaic AI Agent

def ask_agent(question):
    print(f"\n Question: {question}")  
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    answer = response["messages"][-1].content
    print(f"Answer: {answer}")
    return answer

# Test with real questions on Bronze Delta tables!
ask_agent("What products were sold and what is the total sales amount?")
ask_agent("Which users are from which cities?")
ask_agent("What is the status of the last ingestion run?")


 Question: What products were sold and what is the total sales amount?
Answer: The products sold are Laptop, Phone, Tablet, Watch, and TV. The total sales amount is 1200 + 800 + 450 + 300 + 1500 = 4250.

 Question: Which users are from which cities?
Answer: The users are from the following cities: 

- John Doe (user_id 1) is from New York
- Sara Smith (user_id 2) is from London
- Mike Jones (user_id 3) is from Toronto
- Lisa Brown (user_id 4) is from Sydney
- Tom Wilson (user_id 5) is from Singapore

 Question: What is the status of the last ingestion run?
Answer: The status of the last ingestion run was successful, with 5 rows ingested from the customer_api source, starting at 2026-04-19 17:38:24. There were no error messages reported.


'The status of the last ingestion run was successful, with 5 rows ingested from the customer_api source, starting at 2026-04-19 17:38:24. There were no error messages reported.'

In [0]:
# MLflow LLMOps Tracking for Mosaic Agent

import mlflow
import time

mlflow.set_experiment("/Users/theepankumar72@gmail.com/mosaic_agent_llmops")

def ask_agent_with_tracking(question):
    with mlflow.start_run():
        # Log the question
        mlflow.log_param("question", question)
        mlflow.log_param("model", "databricks-meta-llama-3-3-70b-instruct")
        
        # Track time
        start_time = time.time()
        
        # Call agent
        response = agent.invoke({
            "messages": [{"role": "user", "content": question}]
        })
        
        # Calculate duration
        duration = time.time() - start_time
        answer = response["messages"][-1].content
        
        # Log metrics and full answer
        mlflow.log_metric("response_time_seconds", round(duration, 2))
        mlflow.log_metric("answer_length", len(answer))
        mlflow.log_text(answer, "answer.txt")
        
        print(f"\nQuestion: {question}")
        print(f" Answer: {answer}")
        print(f" Response time: {round(duration, 2)}s")
        print(f" MLflow tracking: ")

# Test with tracking!
ask_agent_with_tracking("What is the total sales amount?")
ask_agent_with_tracking("Which ingestion source had the most rows?")


Question: What is the total sales amount?
 Answer: The total sales amount is $5250. This is calculated by adding up the amounts of all the orders: 1200 + 800 + 450 + 300 + 1500 = 5250.
 Response time: 2.26s
 MLflow tracking: 

Question: Which ingestion source had the most rows?
 Answer: The ingestion source with the most rows is customer_api, with 75 rows ingested in one instance and 5 rows in another, totaling 80 rows.
 Response time: 2.03s
 MLflow tracking: 


In [0]:
# MLflow Tracing - Full Agent Observability
import mlflow

mlflow.set_experiment("/Users/theepankumar72@gmail.com/mosaic_agent_llmops")

# Enable automatic tracing for LangChain/LangGraph
mlflow.langchain.autolog()

@mlflow.trace
def ask_agent_traced(question):
    print(f"\n Question: {question}") 
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    answer = response["messages"][-1].content
    print(f" Answer: {answer}")
    return answer

# Test with tracing!
ask_agent_traced("What is the highest selling product?")
ask_agent_traced("How many users are from Asia?")


 Question: What is the highest selling product?
 Answer: The highest selling product is the TV and Laptop, both of which had sales of $1200 and $1500 respectively, but the TV had the highest amount of $1500.

 Question: How many users are from Asia?
 Answer: Based on the provided API users data, there are 2 users from Asia: 

1. Tom Wilson from Singapore (user_id: 5)
2. None other is explicitly mentioned to be from Asia, but Lisa Brown is from Sydney, Australia, which is often culturally associated with Asia, but geographically part of Oceania. 

So, if we consider only the continent of Asia, there is 1 user from Asia. If we consider the broader geographical and cultural region, there might be 2 users.


'Based on the provided API users data, there are 2 users from Asia: \n\n1. Tom Wilson from Singapore (user_id: 5)\n2. None other is explicitly mentioned to be from Asia, but Lisa Brown is from Sydney, Australia, which is often culturally associated with Asia, but geographically part of Oceania. \n\nSo, if we consider only the continent of Asia, there is 1 user from Asia. If we consider the broader geographical and cultural region, there might be 2 users.'

[Trace(trace_id=tr-cd9436eb0afae9eaeaa5a6e7a669094f), Trace(trace_id=tr-19c73d403cd610c0f1b3d08367bb5074)]